# Musanga V2 — Notebook 00: UAV data catalogue, annotation integrity and illumination baseline

**First notebook in the new V2 workflow. Run CPU-only in Google Colab, after selecting a runtime with Google Drive access.** It is read-only with respect to all legacy datasets, models, maps and notebooks.

## Why V2?

Rebuild the Musanga UAV pipeline with auditable dataset provenance, illumination-robust learning and independent spatial validation. **Do not mix UAV-native processing with the later Sentinel-2 grid.** V2 begins with the *existing* georeferenced native-resolution UAV mosaics from the original notebooks; the existing Luki.tif is retained exactly as found. Resampling to satellite grids is deferred until UAV inference and independent validation are complete.

## Legacy data contracts retained

* Original training sites: MOS_1_2, Yoko, Yangambi, Baego; acquisition years: 2024, 2020, 2024, 2023 respectively.
* Luki acquired 2023, original external-only validation from `Annotations/Tree_Annotation_Luki.gpkg` (layer `photointerpretation_inventory`).
* **New, separate Luki evaluation source:** `Annotations/Tree_Annotation_Luki_Nadine.shp`, together with its mandatory shapefile sidecars. It is **combined for evaluation** with additional Musanga crowns identified by `Espece == "Musanga cecropioides"` in `Annotations/Luki_field_inventory.gpkg` layer `couche_top20`, after duplicate/conflict QA. The original `Tree_Annotation_Luki.gpkg` `Nom` inventory is kept separate.
* Crown target: `Class_ID == 1` = Musanga; other numeric classes = non-Musanga, matching KVO_03 and KVO_04. Filter crowns **≥5 m²** for legacy-compatible benchmarks; report before-filter counts too.
* RGB source imagery: three uint8 bands, native georeferencing and original pixel sizes; do not apply raster reprojection or re-georeferencing.
* Five frozen full-resolution U-Net probability maps, where available, are *diagnostic predictions*, never independent ground truth.

## Notebook outputs

`MyDrive/Colab Notebooks/Musanga/KVO_work/V2/00_input_audit/`

Produces a manifest with source file paths and sidecar signatures, raster properties (CRS, bands, dimensions, pixel area, footprint), per-class crown counts, invalid geometry checks, separate Luki source overlap audit, illumination distributions and brightness–predicted-cover relationships. Missing Nadine data is **reported explicitly** and not silently replaced.

## Planned V2 notebooks

01 — reproducible annotation curation and spatial splits; 02 — frozen baseline U-Net and illumination stress; 03 — robust U-Net training with realistic photometric augmentation; 04 — independent LOSO and Luki-specific evaluation; 05 — locked final production and full UAV inference; 06 — validated satellite fraction targets and transfer. Later notebooks will be added only after Notebook 00 confirms the dataset contract.

> **Interpretation warning:** imagery brightness is not a calibrated irradiance measurement, and correlation with frozen Musanga probabilities does not alone establish misclassification.


## 00 — Environment, source definitions and safeguards

Mount Google Drive; establish the five-site native UAV source registry, original versus new Luki labels, and the exclusive V2 output directory. This cell is CPU-only and does not run inference.

In [ ]:
# V2 / 00 — canonical UAV data inventory: READ-ONLY and CPU-only
# Run in Google Colab. No model fitting, image modification, reprojection or writes to legacy outputs.
%pip -q install rasterio geopandas pyogrio shapely scipy matplotlib pandas
from pathlib import Path
import json, hashlib, math, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
from rasterio.windows import Window
from rasterio.features import geometry_mask
from scipy.stats import spearmanr
from IPython.display import display
from google.colab import drive

DRIVE=Path("/content/drive")
if not (DRIVE/"MyDrive").exists(): drive.mount(str(DRIVE))
ROOT=DRIVE/"MyDrive"/"Colab Notebooks"/"Musanga"
GEO=ROOT/"data"/"Geo_Ref"; ANNOT=GEO/"Annotations"
OUT=ROOT/"KVO_work"/"V2"/"00_input_audit"
OUT.mkdir(parents=True,exist_ok=True)
assert GEO.exists(),f"Missing established Geo_Ref directory: {GEO}"

# Historical provenance: KVO_03, KVO_04, and KVO_08 on main.
# Native UAV images are AUTHORITATIVE here. No satellite reprojection, resampling, georeferencing
# or 10-m aggregation in V2/00. In particular, leave the existing Luki.tif exactly as it is.
SITES={
 "MOS_1_2":dict(rgb="MOS_1_2_kvo.tif",year=2024,crs="EPSG:32635",
                training="Tree_Annotations_MOS_1_2_kvo.shp"),
 "Yoko":dict(rgb="Yoko_kvo_georef.tif",year=2020,crs="EPSG:32635",
                training="Tree_Annotations_Yoko_kvo_georef.shp"),
 "Yangambi":dict(rgb="Yangambi_georef.tif",year=2024,crs="EPSG:32635",
                training="Tree_Annotation_Yangambi_georef.shp"),
 "Baego":dict(rgb="Baego_georef.tif",year=2023,crs="EPSG:32635",
                training="Tree_Annotation_Baego.shp"),
 "Luki":dict(rgb="Luki.tif",year=2023,crs="EPSG:32733",
              external_original="Tree_Annotation_Luki.gpkg",
              external_nadine="Tree_Annotation_Luki_Nadine.shp"),
}
# Luki original GPKG has known layer "photointerpretation_inventory" from earlier audit.
# It is kept SEPARATE from Nadine's new shapefile; neither is silently merged or used for training.
LAYERS={"Luki:external_original":"photointerpretation_inventory"}
FIELD_LAYER="couche_top20"
FIELD_GPKG_EXPECTED=ANNOT/"Luki_field_inventory.gpkg"
FIELD_SPECIES_COLUMN="Espece"
FIELD_MUSANGA="Musanga cecropioides"
# Field inventory is a DISTINCT GeoPackage from Tree_Annotation_Luki.gpkg.
MIN_AREA_M2=5.0
MUSANGA_CLASS_ID=1
BLOCK_M=100.0
ILLUM_CANDIDATES=72        # spatially stratified windows/site, not a whole-mosaic pixel census
WINDOW_PX=512
PIXELS_PER_WINDOW=512      # bounded subsample for pixel diagnostics; avoid huge in-memory tables
SEED=20261013
EXCLUDE_BACKGROUND=True
MAPS=ROOT/"KVO_work"/"KVO_04_final_multisite"/"probability_maps"
ILLUM_ARCHIVE=ROOT/"KVO_work"/"KVO_04_final_multisite"/"illumination_audit"

def component_paths(p):
    if p.suffix.lower()==".shp":
        return sorted(x for x in p.parent.glob(p.stem+".*") if x.is_file())
    return [p] if p.exists() else []
def signature(p):
    return [{"path":str(x),"size_bytes":x.stat().st_size,"mtime_ns":x.stat().st_mtime_ns}
            for x in component_paths(p)]
def valid_rgb(rgb):
    return ~(np.all(rgb==0,axis=0)|np.all(rgb==255,axis=0)|
             ((rgb[0]==0)&(rgb[1]==0)&(rgb[2]==255)))
def lum(rgb):
    return (.2126*rgb[0].astype(np.float32)+.7152*rgb[1].astype(np.float32)+
            .0722*rgb[2].astype(np.float32))/255.0
def candidate_windows(h,w,n,side,seed):
    # Sample grid cells with jitter, avoiding over-representing one side of a mosaic.
    rng=np.random.default_rng(seed)
    gy=max(2,int(np.ceil(np.sqrt(n*h/max(w,1)))))
    gx=max(2,int(np.ceil(n/gy)))
    for iy in range(gy):
        for ix in range(gx):
            y0=int(iy*max(0,h-side)/gy);y1=int((iy+1)*max(0,h-side)/gy)+1
            x0=int(ix*max(0,w-side)/gx);x1=int((ix+1)*max(0,w-side)/gx)+1
            yield min(int(rng.integers(y0,max(y0+1,y1))),max(0,h-side)),min(int(rng.integers(x0,max(x0+1,x1))),max(0,w-side))



## 01 — Input dataset registry and native UAV raster properties

Audit exact file locations, RGB bands, acquisition years, dimensions, native pixel sizes, coordinate reference systems and optional baseline probability maps. No imagery is modified.

In [ ]:
# === 1. Full authoritative input/file registry ===
files=[];rasters=[];raster_meta={}
for site, cfg in SITES.items():
    rp=GEO/cfg["rgb"]
    assert rp.exists(),f"Required UAV raster missing for {site}: {rp}"
    files.append(dict(site=site,role="native_uav_rgb",required=True,present=True,
                      path=str(rp),source_metadata=signature(rp)))
    with rasterio.open(rp) as src:
        transform=src.transform
        assert src.count>=3,f"{site}: expected RGB bands"
        assert src.crs and src.crs.is_projected,f"{site}: missing projected CRS"
        expected=cfg["crs"]
        assert src.crs.to_string()==expected,(site,src.crs,expected)
        assert all(dt=="uint8" for dt in src.dtypes[:3]),(site,src.dtypes)
        gsd_x,gsd_y=abs(transform.a),abs(transform.e)
        info=dict(site=site,year=cfg["year"],path=str(rp),width=src.width,height=src.height,
            bands=src.count,dtypes=",".join(src.dtypes),pixel_size_x_m=gsd_x,
            pixel_size_y_m=gsd_y,pixel_area_m2=gsd_x*gsd_y,
            nominal_footprint_km2=src.width*src.height*gsd_x*gsd_y/1e6,
            crs=src.crs.to_string(),transform=str(transform),
            nodata=str(src.nodata),colorinterp=",".join(str(x) for x in src.colorinterp),
            bounds_left=src.bounds.left,bounds_bottom=src.bounds.bottom,
            bounds_right=src.bounds.right,bounds_top=src.bounds.top)
        rasters.append(info);raster_meta[site]=info
    for role in ("training","external_original","external_nadine"):
        if role not in cfg:continue
        p=ANNOT/cfg[role]
        files.append(dict(site=site,role=role,required=True,present=p.exists(),
                          path=str(p),source_metadata=signature(p)))
        if not p.exists():
            print("MISSING ANNOTATION:",site,role,p)
    pp=MAPS/f"{site}_musanga_probability_uint16_FULL.tif"
    files.append(dict(site=site,role="frozen_probability_optional",required=False,
                      present=pp.exists(),path=str(pp),source_metadata=signature(pp)))
# Exact established file/layer from the historical Luki annotation workflow.
# Only inspect the known canonical source; don't search unrelated GPKGs.
FIELD_GPKG=None
if FIELD_GPKG_EXPECTED.exists():
    layers=gpd.list_layers(FIELD_GPKG_EXPECTED)
    print("Luki field inventory available layers:",list(layers["name"]))
    if FIELD_LAYER in set(layers["name"].astype(str)):
        FIELD_GPKG=FIELD_GPKG_EXPECTED
        print("Luki field inventory resolved:",FIELD_GPKG,"layer:",FIELD_LAYER)
    else:
        print("ERROR: canonical Luki GPKG found, but expected couche_top20 layer missing")
else:
    print("ERROR: canonical Luki field inventory not found:",FIELD_GPKG_EXPECTED)
files.append(dict(site="Luki",role="external_field_inventory_musanga",
    required=True,present=FIELD_GPKG is not None,
    path=str(FIELD_GPKG) if FIELD_GPKG else "UNRESOLVED:"+FIELD_LAYER,
    source_metadata=signature(FIELD_GPKG) if FIELD_GPKG else []))
FILE_DF=pd.DataFrame(files);RASTER_DF=pd.DataFrame(rasters)
FILE_DF.to_json(OUT/"00_input_manifest.json",orient="records",indent=2)
RASTER_DF.to_csv(OUT/"00_uav_raster_metadata.csv",index=False)
print("Native raster inventory (no changes to pixels or georeferencing):")
display(RASTER_DF[["site","year","width","height","bands","pixel_size_x_m",
                   "pixel_size_y_m","nominal_footprint_km2","crs","nodata"]].round(4))
display(FILE_DF[["site","role","present","path"]])
if not (ANNOT/"Tree_Annotation_Luki_Nadine.shp").exists():
    print("ACTION NEEDED: put Nadine's Tree_Annotation_Luki_Nadine.shp AND sidecars (.shx, .dbf, .prj)")
    print("in",ANNOT,"and rerun. No surrogate Luki annotation will be used.")



## 02 — Crown annotation counts and consistency checks

Inspect numeric `Class_ID` annotations and the original Luki GeoPackage's **textual `Nom` labels** separately. Preserve raw labels; do not guess species mapping. Flag invalid geometries and apply ≥5 m² compatibility rule.


In [ ]:
# === 2. Class/polygon QA, identical >=5 m2 definition as KVO_03 and KVO_04 ===
ANNSETS={};QC=[];CLASSES=[];COUNTS=[]
for site,cfg in SITES.items():
    src_crs=rasterio.crs.CRS.from_string(cfg["crs"])
    for role in ("training","external_original","external_nadine"):
        if role not in cfg:continue
        ap=ANNOT/cfg[role]
        if not ap.exists():continue
        kwargs={"layer":LAYERS[f"{site}:{role}"]} if f"{site}:{role}" in LAYERS else {}
        try:g=gpd.read_file(ap,**kwargs)
        except Exception as e:
            print(f"ERROR reading {site}/{role}: {type(e).__name__}: {e}")
            continue
        # Luki original is an inventory with a textual 'Nom' field, not Class_ID.
        # Never infer numeric species IDs from arbitrary names: preserve the full label inventory.
        if "Class_ID" in g.columns:
            label_field="Class_ID"; numeric_labels=True
        elif site=="Luki" and role=="external_original" and "Nom" in g.columns:
            label_field="Nom"; numeric_labels=False
        else:
            print(f"ERROR: {site}/{role}: unknown classification fields: {list(g.columns)}")
            continue
        for value,count in g[label_field].fillna("<missing>").astype(str).value_counts().items():
            COUNTS.append(dict(site=site,source=role,label_field=label_field,
                               raw_label=value,count=int(count)))
        native_crs=str(g.crs)
        if g.crs is None:
            print("ERROR: missing annotation CRS; not guessing:",site,role)
            continue
        # CRS transformation is IN MEMORY and vectors only, to measure m² and overlay the
        # CURRENT existing UAV raster. No source file or raster is ever reprojected.
        aligned=(g.crs==src_crs)
        if not aligned:g=g.to_crs(src_crs)
        g=g[g.geometry.notna() & ~g.geometry.is_empty].copy()
        raw_n=len(g)
        if numeric_labels:
            numeric=pd.to_numeric(g["Class_ID"],errors="coerce")
            invalid_class=int(numeric.isna().sum())
            g=g[numeric.notna()].copy()
            g["class_id"]=numeric.loc[g.index].astype(int)
            g["original_label"]=g["Class_ID"].astype(str)
        else:
            invalid_class=0
            g["class_id"]=pd.Series(pd.NA,index=g.index,dtype="Int64")
            g["original_label"]=g["Nom"].fillna("<missing>").astype(str)
        # Textual labels are preserved, not force-mapped into Musanga/non-Musanga.
        g["area_m2"]=g.geometry.area.astype(float)
        # Explicit unknown/other class accounting; previous notebooks used Class_ID==1 as positive.
        bad_geom=int((~g.geometry.is_valid).sum())
        bbox=rasterio.coords.BoundingBox(
            raster_meta[site]["bounds_left"],raster_meta[site]["bounds_bottom"],
            raster_meta[site]["bounds_right"],raster_meta[site]["bounds_top"])
        bounds_poly=__import__("shapely.geometry",fromlist=["box"]).box(*bbox)
        g["intersects_rgb"]=g.geometry.intersects(bounds_poly)
        kept=g[(g.area_m2>=MIN_AREA_M2)&g.intersects_rgb].copy()
        # Stable across reruns for now: input source feature index, NOT merged between Luki sources.
        kept["source_row"]=kept.index.astype(str)
        kept["is_musanga"]=(kept.class_id==MUSANGA_CLASS_ID) if numeric_labels else pd.Series(pd.NA,index=kept.index,dtype="boolean")
        kept["source_role"]=role
        ANNSETS[(site,role)]=kept
        if numeric_labels:
            for cl,ct in g.class_id.value_counts(dropna=False).items():
                CLASSES.append(dict(site=site,source=role,class_id=int(cl),count=int(ct)))
        q=dict(site=site,source=role,path=str(ap),native_annotation_crs=native_crs,
             raster_crs=src_crs.to_string(),vector_reprojected_in_memory=not aligned,
             classification_field=label_field,
             classification_status="numeric" if numeric_labels else "text_unmapped",
             original_features=raw_n,invalid_class=invalid_class,invalid_geometry=bad_geom,
             labelled_features=len(g),outside_uav_footprint=int((~g.intersects_rgb).sum()),
             under_5m2=int((g.area_m2<MIN_AREA_M2).sum()),
             n_after_area_and_footprint=len(kept),
             musanga_n=int(kept.is_musanga.sum()) if numeric_labels else np.nan,
             non_musanga_n=int((~kept.is_musanga).sum()) if numeric_labels else np.nan,
             musanga_pct=float(100*kept.is_musanga.mean()) if numeric_labels and len(kept) else np.nan,
             median_area_m2=float(kept.area_m2.median()) if len(kept) else np.nan,
             sum_polygon_area_m2=float(kept.area_m2.sum()),
             bounds_overlap_pct=float(100*g.intersects_rgb.mean()) if len(g) else np.nan)
        QC.append(q)
        print("AUDITED:",site,role,"eligible crowns:",len(kept),
              "Musanga:",q["musanga_n"],"other:",q["non_musanga_n"],"classification:",q["classification_status"])
QC_DF=pd.DataFrame(QC)
if len(QC_DF):
    QC_DF.to_csv(OUT/"00_annotation_quality_and_counts.csv",index=False)
    display(QC_DF[["site","source","original_features","under_5m2",
                   "n_after_area_and_footprint","musanga_n","non_musanga_n",
                   "vector_reprojected_in_memory","invalid_geometry"]])
pd.DataFrame(CLASSES).to_csv(OUT/"00_class_id_distribution.csv",index=False)
LABEL_DF=pd.DataFrame(COUNTS)
LABEL_DF.to_csv(OUT/"00_raw_annotation_label_inventory.csv",index=False)
print("Source label inventory (especially original Luki Nom):")
display(LABEL_DF[LABEL_DF.site=="Luki"])



## 03 — Luki evaluation: versioned Nadine + field-inventory Musanga crowns

The Luki field inventory is **not** the original `Tree_Annotation_Luki.gpkg` photointerpretation file. Read `data/Geo_Ref/Annotations/Luki_field_inventory.gpkg` containing layer `couche_top20`; select only records whose `Espece` is `Musanga cecropioides`. Combine these Musanga polygons with Nadine's binary-crown shapefile. Nadine supplies both positive and negative observations; the field inventory contributes only species-confirmed positives.

Keep original annotation sources unchanged. Field/Nadine duplicates are removed by spatial overlap; contradictory labels are flagged and both affected evaluation annotations excluded pending review. Save the derived evaluation GeoPackage, overlap decisions and manifest to **`Musanga/KVO_work/V2/00_input_audit/`**. The older photointerpretation `Nom` file remains a separate legacy inventory.


On rerun, the previous merged 364-crown reference is archived by content hash before the current merged pointer is replaced. New dark-area observations remain exclusively external; the four development forests do not change.


In [ ]:
# === 3. Luki-specific two-source distinction and possible spatial overlap ===
# Do NOT union these geometries: duplicate/overlapping annotations can leak across folds.
LUKI_OVERLAP=None
k0=("Luki","external_original");k1=("Luki","external_nadine")
if k0 in ANNSETS and k1 in ANNSETS:
    old=ANNSETS[k0];new=ANNSETS[k1]
    if len(old) and len(new):
        from shapely.strtree import STRtree
        old_geoms=list(old.geometry)
        tree=STRtree(old_geoms)
        rows=[]; radius=10.0
        # Shapely 2 returns integer indices for candidate geometries.
        for ix,geom in enumerate(new.geometry):
            candidates=tree.query(geom.buffer(radius))
            overlap=0.;near=False;hits=0;oldclasses=set()
            for jj in candidates:
                j=int(jj);other=old_geoms[j]
                dist=float(geom.distance(other))
                if dist<=radius:near=True
                if geom.intersects(other):
                    hits+=1
                    overlap+=float(geom.intersection(other).area)
                    oldclasses.add(str(old.iloc[j].original_label))
            rows.append(dict(nadine_source_row=str(new.iloc[ix].source_row),
                nadine_class_id=int(new.iloc[ix].class_id),
                overlaps_original_count=hits,
                overlap_area_m2=overlap,
                overlap_fraction_nadine=min(1.,overlap/max(float(geom.area),1e-10)),
                within_10m_of_original=near,
                overlap_original_labels=" | ".join(sorted(oldclasses))))
        LUKI_OVERLAP=pd.DataFrame(rows)
        LUKI_OVERLAP.to_csv(OUT/"00_luki_nadine_vs_original_overlap.csv",index=False)
        print("LUKI cross-source overlap: Nadine crowns overlapping original:",
              int((LUKI_OVERLAP.overlaps_original_count>0).sum()),"/",len(LUKI_OVERLAP))
        print("Nadine crowns within 10 m of original annotation:",int(LUKI_OVERLAP.within_10m_of_original.sum()))
        # Only a potential leakage diagnostic, NOT a label/conflict resolver.
elif k1 not in ANNSETS:
    print("Luki new Nadine annotation unavailable or invalid; overlap QA pending.")


# === 3B. Build Nadine + field-inventory Musanga evaluation reference ===
# Source policy: Nadine is the independent binary crown annotation; the field
# inventory layer couche_top20 adds ONLY polygons with Espece == 'Musanga cecropioides'.
# Never infer non-Musanga from other Espece values or photointerpretation 'Nom'.
# No imagery or original polygons are edited; only derived outputs written to OUT.
LUKI_MERGED=None
if FIELD_GPKG is None or ("Luki","external_nadine") not in ANNSETS:
    print("MERGE NOT PRODUCED: requires uniquely identified field-inventory GPKG and Nadine polygons.")
else:
    from shapely.strtree import STRtree
    try:
        field_raw=gpd.read_file(FIELD_GPKG,layer=FIELD_LAYER)
        assert FIELD_SPECIES_COLUMN in field_raw.columns, (
            f"{FIELD_GPKG} / {FIELD_LAYER}: missing {FIELD_SPECIES_COLUMN}; found {list(field_raw.columns)}")
        assert field_raw.crs is not None,"Field inventory has no CRS. Do not infer one."
        print("Field inventory rows:",len(field_raw))
        species_counts=field_raw[FIELD_SPECIES_COLUMN].fillna("<missing>").astype(str).value_counts()
        species_counts.rename_axis("Espece").reset_index(name="n").to_csv(
            OUT/"00_luki_field_inventory_species.csv",index=False)
        display(species_counts.head(25).rename("count").to_frame())
        species=field_raw[FIELD_SPECIES_COLUMN].fillna("").astype(str).str.strip()
        field=field_raw.loc[species.str.casefold()==FIELD_MUSANGA.casefold()].copy()
        n_species=len(field)
        print("Confirmed Musanga cecropioides in field inventory:",n_species)
        target_crs=rasterio.crs.CRS.from_string(SITES["Luki"]["crs"])
        if field.crs!=target_crs:field=field.to_crs(target_crs)
        field=field[field.geometry.notna()&~field.geometry.is_empty].copy()
        if not field.empty:
            invalid=int((~field.geometry.is_valid).sum())
            if invalid:print("WARNING: invalid field Musanga polygons:",invalid,"(retained in audit, not auto-repaired)")
        # Only valid geometries are safe for intersection / de-duplication.
        field=field[field.geometry.is_valid].copy()
        field["area_m2"]=field.geometry.area
        before_area=len(field)
        field=field[field.area_m2>=MIN_AREA_M2].copy()
        bounds=raster_meta["Luki"]
        from shapely.geometry import box
        footprint=box(bounds["bounds_left"],bounds["bounds_bottom"],
                      bounds["bounds_right"],bounds["bounds_top"])
        field=field[field.geometry.intersects(footprint)].copy()
        field["source_id"]=field.index.astype(str)
        nadine=ANNSETS[("Luki","external_nadine")].copy()
        # Newly digitized features can contain ring self-intersections.
        # Never mutate the source SHP or silently repair crown shapes:
        # audit each invalid record by its original input row ID, then exclude
        # ONLY invalid polygons from this derived reference and overlap matching.
        from shapely.validation import explain_validity
        invalid_mask=~nadine.geometry.is_valid
        invalid_nadine=int(invalid_mask.sum())
        if invalid_nadine:
            invalid_rows=nadine.loc[invalid_mask].copy()
            invalid_report=pd.DataFrame([
                dict(source_dataset="nadine",source_row=str(rec.source_row),
                     class_id=int(rec.class_id),area_m2=float(rec.area_m2),
                     validity_reason=explain_validity(rec.geometry),
                     action="excluded_from_derived_reference_source_unchanged")
                for _,rec in invalid_rows.iterrows()])
            invalid_report.to_csv(OUT/"00_Luki_Nadine_invalid_polygons.csv",index=False)
            print("WARNING:",invalid_nadine,"invalid Nadine crown(s) excluded from derived Luki evaluation:")
            display(invalid_report)
            nadine=nadine.loc[~invalid_mask].copy()
        else:
            print("All",len(nadine),"eligible Nadine crowns have valid geometry.")
        assert nadine.geometry.is_valid.all()
        assert nadine.source_row.astype(str).is_unique, "Nadine source row IDs must be unique"
        print("Valid Nadine crowns available to merge:",len(nadine))
        # Distinct source annotations must not be flattened before conflicts are identified.
        ng=list(nadine.geometry)
        ntree=STRtree(ng)
        decisions=[];keep_field=[];conflict_nadine=set()
        # Conservative matching: spatial intersection >=30% of either crown area.
        # Marginal intersects are recorded for review, not automatically removed.
        DUPLICATE_OVERLAP=0.30
        for idx,rec in field.iterrows():
            geom=rec.geometry
            duplicates=[];touches=[]
            for pos in ntree.query(geom):
                jj=int(pos);other=ng[jj]
                ia=geom.intersection(other).area
                fraction=ia/max(min(geom.area,other.area),1e-9)
                if ia>0 and fraction>=DUPLICATE_OVERLAP:
                    duplicates.append((jj,float(fraction)))
                elif ia>0:
                    touches.append((jj,float(fraction)))
            labels=sorted(set(int(nadine.iloc[jj].class_id) for jj,_ in duplicates))
            if not duplicates:action="add_field_musanga";keep_field.append(idx)
            elif labels==[1]:action="duplicate_nadine_musanga"
            else:
                action="conflicting_label_exclude_both"
                conflict_nadine.update(jj for jj,_ in duplicates)
            decisions.append(dict(field_row=str(idx),field_species=FIELD_MUSANGA,
                action=action,matched_nadine_count=len(duplicates),
                matched_nadine_class_ids=",".join(map(str,labels)),
                max_matched_overlap_fraction=max([v for _,v in duplicates],default=0),
                slight_overlap_nadine_count=len(touches),
                source_file=str(FIELD_GPKG),source_layer=FIELD_LAYER))
        decision_df=pd.DataFrame(decisions,columns=["field_row","field_species","action",
            "matched_nadine_count","matched_nadine_class_ids","max_matched_overlap_fraction",
            "slight_overlap_nadine_count","source_file","source_layer"])
        decision_df.to_csv(OUT/"00_luki_field_to_nadine_overlap_decisions.csv",index=False)
        # If any disagreement, exclude both annotations from scientific evaluation;
        # retain complete conflict record for manual adjudication.
        nd_keep=nadine.iloc[[i for i in range(len(nadine)) if i not in conflict_nadine]].copy()
        fg_keep=field.loc[keep_field].copy()
        nd_merge=gpd.GeoDataFrame(dict(
            source_dataset=["nadine"]*len(nd_keep),
            source_id=nd_keep.source_row.astype(str).to_numpy(),
            class_id=nd_keep.class_id.astype(int).to_numpy(),
            class_name=np.where(nd_keep.is_musanga.to_numpy(),"Musanga","Non-Musanga"),
            area_m2=nd_keep.area_m2.to_numpy()),geometry=nd_keep.geometry.to_numpy(),crs=target_crs)
        fg_merge=gpd.GeoDataFrame(dict(
            source_dataset=["field_inventory"]*len(fg_keep),
            source_id=fg_keep.source_id.astype(str).to_numpy(),
            class_id=np.ones(len(fg_keep),dtype=int),
            class_name=["Musanga"]*len(fg_keep),
            area_m2=fg_keep.area_m2.to_numpy()),geometry=fg_keep.geometry.to_numpy(),crs=target_crs)
        # Spatially overlapping FIELD crowns may also duplicate each other.
        # Audit separately and exclude ambiguous self-duplicates deterministically.
        if len(fg_merge)>1:
            to_drop=set();polys=list(fg_merge.geometry);ptree=STRtree(polys)
            for ii,geom in enumerate(polys):
                if ii in to_drop:continue
                for jj0 in ptree.query(geom):
                    jj=int(jj0)
                    if jj<=ii:continue
                    ov=geom.intersection(polys[jj]).area/max(min(geom.area,polys[jj].area),1e-9)
                    if ov>=DUPLICATE_OVERLAP:to_drop.add(jj)
            if to_drop:
                print("Field self-overlapping Musanga crowns removed:",len(to_drop))
                fg_merge=fg_merge.iloc[[i for i in range(len(fg_merge)) if i not in to_drop]].copy()
        LUKI_MERGED=gpd.GeoDataFrame(pd.concat([nd_merge,fg_merge],ignore_index=True),crs=target_crs)
        assert len(LUKI_MERGED)>0
        path=OUT/"00_Luki_merged_evaluation_crowns.gpkg"
        # This is a new V2 derived output; never overwrite legacy source files.
        # Preserve the previous benchmark BEFORE replacing the latest pointer.
        # A SHA256 byte hash identifies each frozen merged reference version.
        import shutil
        archive=OUT/"luki_reference_versions"
        archive.mkdir(parents=True,exist_ok=True)
        if path.exists():
            previous_hash=hashlib.sha256(path.read_bytes()).hexdigest()[:16]
            previous=archive/f"Luki_reference_{previous_hash}.gpkg"
            if not previous.exists():
                shutil.copy2(path,previous)
                print("ARCHIVED PREVIOUS LUKI REFERENCE:",previous)
            path.unlink()
        LUKI_MERGED.to_file(path,layer="evaluation_crowns",driver="GPKG")
        current_hash=hashlib.sha256(path.read_bytes()).hexdigest()[:16]
        current_archive=archive/f"Luki_reference_{current_hash}.gpkg"
        if not current_archive.exists():shutil.copy2(path,current_archive)
        (OUT/"00_luki_current_reference_version.json").write_text(
            json.dumps(dict(reference_id=current_hash,
                frozen_archive=str(current_archive),current_pointer=str(path),
                n_crowns=len(LUKI_MERGED),
                musanga=int((LUKI_MERGED.class_id==1).sum()),
                other=int((LUKI_MERGED.class_id!=1).sum()),
                nadine_source_signatures=signature(ANNOT/SITES["Luki"]["external_nadine"]),
                field_source_signatures=signature(FIELD_GPKG)),indent=2))
        print("LUKI CURRENT REFERENCE VERSION:",current_hash)
        print("IMMUTABLE COPY:",current_archive)
        print("\nMERGED LUKI EVALUATION created:",path)
        print("Nadine retained:",len(nd_merge),"| field Musanga additions:",len(fg_merge))
        print("Field duplicates of Nadine:",sum(decision_df.action=="duplicate_nadine_musanga"))
        print("Field/Nadine label conflicts:",sum(decision_df.action=="conflicting_label_exclude_both"))
        print("Conflicting Nadine crowns excluded:",len(conflict_nadine))
        print("TOTAL:",len(LUKI_MERGED),"Musanga:",
              int((LUKI_MERGED.class_id==1).sum()),"other:",
              int((LUKI_MERGED.class_id!=1).sum()))
        summary=dict(field_gpkg=str(FIELD_GPKG),field_layer=FIELD_LAYER,
          species_column=FIELD_SPECIES_COLUMN,musanga_name=FIELD_MUSANGA,
          field_species_matches=n_species,field_after_area=before_area,
          eligible_field_musanga=len(field),
          matched_field_duplicates=int(sum(decision_df.action=="duplicate_nadine_musanga")),
          field_conflicts=int(sum(decision_df.action=="conflicting_label_exclude_both")),
          nadine_conflicts_excluded=len(conflict_nadine),
          nadine_retained=len(nd_merge),nadine_invalid_excluded=invalid_nadine,field_added=len(fg_merge),
          merged_total=len(LUKI_MERGED),
          merged_musanga=int((LUKI_MERGED.class_id==1).sum()),
          merged_other=int((LUKI_MERGED.class_id!=1).sum()),
          overlap_fraction_threshold=DUPLICATE_OVERLAP,
          spatial_crs=str(target_crs),
          policy="Do not use Luki evaluation polygons as training labels in V2 external validation")
        (OUT/"00_luki_merged_evaluation_manifest.json").write_text(json.dumps(summary,indent=2))
        display(pd.DataFrame([summary]).T.rename(columns={0:"value"}))
    except Exception as error:
        print("MERGE FAILED, no silent fallback:",type(error).__name__,str(error))
        raise


## 04 — UAV illumination and predicted Musanga cover

Characterize sampled RGB luminance and its association with the frozen U-Net probabilities for all sites. Brightness is observational, not calibrated irradiance; associations are not proof of incorrect species identification.

In [ ]:
# === 4. Illumination from native UAV RGB and frozen probabilities (when available) ===
# All stats here are SAMPLE ESTIMATES, not a census; annotate sampled pixel/window counts.
# Use source-derived RGB validity sentinels as in KVO_04 production, not Luki band 4 mask.
LIGHT=[];PIXELS=[];CAND=[]
for j,(site,cfg) in enumerate(SITES.items()):
    rgbpath=GEO/cfg["rgb"];probpath=MAPS/f"{site}_musanga_probability_uint16_FULL.tif"
    with rasterio.open(rgbpath) as src:
        prob=rasterio.open(probpath) if probpath.exists() else None
        if prob is not None:
            if not (prob.width==src.width and prob.height==src.height and prob.transform==src.transform and prob.crs==src.crs):
                print("WARNING: misaligned probability map, ignoring",site)
                prob.close();prob=None
        rng=np.random.default_rng(SEED+j)
        for row,col in candidate_windows(src.height,src.width,ILLUM_CANDIDATES,WINDOW_PX,SEED+j):
            h=min(WINDOW_PX,src.height-row);w=min(WINDOW_PX,src.width-col)
            win=Window(col,row,w,h)
            rgb=src.read([1,2,3],window=win)
            valid=valid_rgb(rgb)
            if valid.mean()<.75:continue
            l=lum(rgb)
            raw=prob.read(1,window=win) if prob else None
            if raw is not None:valid &= raw!=65535
            if valid.mean()<.75:continue
            lv=l[valid]
            rec=dict(site=site,row=row,col=col,valid_pixel_count=int(valid.sum()),
                brightness_mean=float(lv.mean()),brightness_median=float(np.median(lv)),
                brightness_p10=float(np.quantile(lv,.1)),
                brightness_p90=float(np.quantile(lv,.9)),
                brightness_sd=float(lv.std()),
                pct_luminance_below_0p16=float(100*np.mean(lv<.16)))
            if raw is not None:
                p=raw[valid].astype(np.float32)/10000.
                rec.update(mean_probability=float(p.mean()),
                           predicted_musanga_cover=float(np.mean(p>=.5)),
                           prob_brightness_spearman_rho=float(spearmanr(lv[::64],p[::64]).statistic))
            CAND.append(rec)
            # Lightweight random pixels used for brightness histograms and diagnostic scatter.
            flat=np.flatnonzero(valid)
            count=min(PIXELS_PER_WINDOW,len(flat))
            take=rng.choice(flat,count,replace=False)
            pp=raw.ravel()[take]/10000. if raw is not None else np.full(count,np.nan)
            PIXELS.extend(dict(site=site,brightness=float(q),probability=float(p))
                          for q,p in zip(l.ravel()[take],pp))
        if prob:prob.close()
    sub=pd.DataFrame([r for r in CAND if r["site"]==site])
    if len(sub):
        correl=float(spearmanr(sub.brightness_median,sub.predicted_musanga_cover).statistic) if "predicted_musanga_cover" in sub and len(sub)>2 else np.nan
        LIGHT.append(dict(site=site,n_windows=len(sub),illumination_qa_status="LOW_SAMPLE" if len(sub)<20 else "SAMPLED",median_window_brightness=float(sub.brightness_median.median()),
            p10_window_brightness=float(sub.brightness_median.quantile(.1)),
            p90_window_brightness=float(sub.brightness_median.quantile(.9)),
            mean_dark_pixel_pct=float(sub.pct_luminance_below_0p16.mean()),
            tile_brightness_vs_musanga_rho=correl if len(sub)>=20 else np.nan,
            prob_map_available=probpath.exists()))
        print("ILLUMINATION:",site,"windows",len(sub),"rho",round(correl,3) if np.isfinite(correl) and len(sub)>=20 else "INSUFFICIENT WINDOWS",flush=True)
CAND_DF=pd.DataFrame(CAND);LIGHT_DF=pd.DataFrame(LIGHT)
PIXEL_DF=pd.DataFrame(PIXELS)
CAND_DF.to_csv(OUT/"00_illumination_window_samples.csv",index=False)
LIGHT_DF.to_csv(OUT/"00_illumination_summary.csv",index=False)
PIXEL_DF.to_csv(OUT/"00_illumination_pixel_samples.csv",index=False)
display(LIGHT_DF.round(3))
fig,axs=plt.subplots(1,2,figsize=(14,5),constrained_layout=True)
for site in SITES:
    pp=PIXEL_DF.loc[PIXEL_DF.site==site,"brightness"]
    if len(pp):axs[0].hist(pp,bins=40,histtype="step",density=True,label=site)
    dd=CAND_DF[CAND_DF.site==site]
    if len(dd) and "predicted_musanga_cover" in dd:
        axs[1].scatter(dd.brightness_median,dd.predicted_musanga_cover,
                       alpha=.45,s=14,label=site)
axs[0].set(xlabel="Native RGB luminance",ylabel="Sample density",
           title="Illumination distributions by site")
axs[1].set(xlabel="Window median RGB luminance",ylabel="Frozen U-Net predicted Musanga fraction",
           title="Image illumination versus model prediction")
for ax in axs:ax.legend(fontsize=8)
fig.savefig(OUT/"00_illumination_comparison.png",dpi=160,bbox_inches="tight")
display(fig);plt.close(fig)



## 05 — V2 input contract and quality-control summary

Save the provenance and modelling contract, report missing required sources, and confirm that all outputs stay in `Musanga/KVO_work/V2/00_input_audit/`.

In [ ]:
# === 5. Explicit V2 scientific contract: keep datasets and splits unmodified ===
POLICY={
 "project":"Musanga V2 (UAV-first)",
 "native_uav_sources":{site:str(GEO/cfg["rgb"]) for site,cfg in SITES.items()},
 "training_label_sources":{site:str(ANNOT/cfg["training"]) for site,cfg in SITES.items() if "training" in cfg},
 "luki_external_original":str(ANNOT/SITES["Luki"]["external_original"]),
 "luki_external_nadine":str(ANNOT/SITES["Luki"]["external_nadine"]),
 "luki_annotation_policy":"Evaluation merges Nadine Class_ID crowns + Espece == Musanga cecropioides from field inventory couche_top20; de-duplicate, exclude cross-source label conflicts; do not merge photointerpretation Nom file",
 "luki_field_inventory_path":str(FIELD_GPKG) if FIELD_GPKG else None,
 "luki_field_inventory_layer":FIELD_LAYER,
 "luki_merged_evaluation_output":str(OUT/"00_Luki_merged_evaluation_crowns.gpkg") if LUKI_MERGED is not None else None,
 "luki_raster_policy":"Use existing Geo_Ref/Luki.tif as native UAV source; no re-georeferencing or S2 resampling in V2 UAV notebooks",
 "positive_class":"Class_ID == 1; all other numeric Class_ID values are non-Musanga as in KVO_03/04",
 "crown_area_filter_m2":MIN_AREA_M2,
 "minimum_area_filter_policy":"Apply >=5m² for comparable benchmark counts; also retain raw per-class counts.",
 "legacy_original_training_sites":["MOS_1_2","Yoko","Baego","Yangambi"],
 "original_luki_role":"External test only; future inclusion in training requires explicit revised split",
 "illumination":"Descriptive native RGB luminance, uncalibrated; sampled window summaries, not full-mosaic map",
 "reference_probabilities":"Optional frozen baseline, never used as ground-truth labels",
 "annotation_transform":"Vector CRS adjusted IN MEMORY solely for spatial comparisons; files unchanged",
 "outputs":str(OUT),
}
(OUT/"00_v2_data_contract.json").write_text(json.dumps(POLICY,indent=2))
missing=[r for r in files if r["required"] and not r["present"]]
unmapped=QC_DF[QC_DF.classification_status=="text_unmapped"] if len(QC_DF) else pd.DataFrame()
print("\nV2 DATA CONTRACT SAVED. Outputs:",OUT)
if missing:
    print("INCOMPLETE REQUIRED SOURCES — review:")
    for r in missing:print(" -",r["site"],r["role"],r["path"])
else:print("All declared required FILES found (this does not guarantee label compatibility).")
if len(unmapped):
    print("LABEL MAPPING PENDING:",list(zip(unmapped.site,unmapped.source)))
    print("Inspect 00_raw_annotation_label_inventory.csv before interpreting original Luki labels.")
print("No training, source-file writes, raster reprojection, or model inference performed.")
